# Amazon Business Entity Resolution — V8 best-effort GPU/CPU run

This is a **new run**. It does not touch a previously running Kaggle session. Use this notebook in the second account, attach the official student-resource dataset through **Add Input**, choose a Kaggle GPU accelerator, then run top to bottom.

The workflow indexes the complete Source 2 and Source 3 tables and processes every Source 1 training entity. It reserves a stable 1% entity holdout for honest threshold/model selection, then includes that holdout in the final production refit under the same positive/hard-negative sampling policy. Only easy negative *pairs* are downsampled; positive pairs are kept. The GPU accelerates CatBoost fitting and, after a runtime probe, numeric-only CatBoost prediction; LightGBM remains CPU-based. Blocking and pair-feature generation remain CPU work and are expected to dominate wall time.

V8 fixes material issues found in the code review: (1) it adds standalone rare address-token and transliterated-address-token blocking channels, because the V6.1 audit attributed much of its recovered recall to address evidence even when names were corrupted; (2) inference predicts in batches, instead of making one model call for every single query row; (3) training pair generation and test inference checkpoint durable batches so a kernel interruption does not force a full rebuild when Kaggle preserves `/kaggle/working`. It compares GPU CatBoost with CPU LightGBM on the same entity holdout, tries a small blend-weight grid, optimizes source-specific thresholds for the actual per-entity macro F0.5, and refits the selected model/blend using every generated labeled pair.

## Read this before spending the long run

- **99 F0.5 is a target, not a promise.** The V6.1 `0.978` is an oracle ceiling on a 5,000-row prefix-derived validation slice, not a model score or leaderboard result. Its old validation selection used the first 25,000 Source-1 rows before shuffling, so it is not a reliable estimate of the hidden test distribution. If this notebook's held-out candidate-only oracle is below `0.99`, no classifier or ensemble can reach `0.99` with this candidate set; retrieval would have to improve.
- F0.5 is not ordinary accuracy. It is entity-level macro F0.5 with a strong precision preference and singleton protection. The notebook reports both candidate oracle and model F0.5 so blocking failures cannot be mistaken for classifier failures.
- Candidate cap is 128 **per target source**. The cap is evaluated on a stable, entity-level holdout using the same source data and feature rules as test. Raising it improves possible recall but increases full test scoring time; the cap/oracle report is the evidence for deciding whether the budget is worthwhile.
- Training checkpoints are written every 50,000 Source-1 rows per target source; CatBoost also saves GPU training snapshots every five minutes. Re-run the training cell after a kernel interruption; it rebuilds the current target index, then continues from the last committed row. Test inference likewise resumes from its latest committed 1,024-query batch. A completely new Kaggle session may not retain `/kaggle/working`, so save/attach notebook outputs if you need to carry checkpoints across sessions.
- This is a feature-based entity matcher, not an LLM pairwise scorer. A transformer cross-encoder can be strong on entity-matching benchmarks, but scoring hundreds of millions of candidate pairs with it is not a practical Kaggle T4 plan. Published scores on other datasets do not transfer as a guaranteed score here.

## Why this model mix

The input to the matcher is 60+ numeric pair comparisons, token/address/character similarities, source identity, and independent blocking provenance. Boosted trees are a good fit for this structured pair evidence. CatBoost has a documented multi-GPU training path; this notebook uses both Kaggle GPUs when available. LightGBM is the lower-overhead CPU comparator; its full CUDA build requires a separate source build, so installing the ordinary wheel does not mean it is using the T4s. The validation set decides whether CatBoost, LightGBM, or their probability blend wins.

Research/reference notes: [Ditto entity matching paper](https://arxiv.org/abs/2004.00584) demonstrates transformer pair classification and difficult-example augmentation, while its results are on separate benchmark datasets; the [blocking/filtering survey](https://arxiv.org/abs/1905.06167) describes the recall/efficiency trade-off that creates the candidate ceiling. Official implementation guidance: [CatBoost multi-GPU training](https://catboost.ai/docs/en/features/training-on-gpu), [CatBoost speed settings](https://catboost.ai/docs/en/concepts/speed-up-training), [CatBoost numeric-only GPU prediction API](https://github.com/catboost/catboost/blob/master/catboost/docs/en/concepts/python-reference_catboost_predict.md), and [LightGBM GPU build requirements](https://lightgbm.readthedocs.io/en/stable/Installation-Guide.html#build-cuda-version). Public challenge repositories provide useful but unverified comparison points: [one self-reports 0.976105 validation macro F0.5 with an LGBM + HistGradientBoosting blend](https://github.com/Akash-bardia/amazon-ml-challenge-2026), and [another outlines multi-view TF-IDF retrieval with cross-fitted two-stage boosting but expects a 32-vCPU/128-GB host](https://github.com/AyanAhmedKhan/amazon-ml-challenge). These do not establish a leaderboard score or make their resource profile portable to this Kaggle session.


In [ ]:
from pathlib import Path
import base64, io, zipfile

REPO_DIR = Path('/kaggle/working/AMAZON-ML-CHALLENGE-V8')
REPO_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = """
...
cHlQSwUGAAAAAAUABQDLAQAAQ3kAAAAA
"""
raw_bundle = base64.b64decode(''.join(BUNDLE_B64.split()))
with zipfile.ZipFile(io.BytesIO(raw_bundle)) as bundle:
    bundle.extractall(REPO_DIR)
print('Embedded V8 scripts extracted to', REPO_DIR)
print('Embedded files:', len(bundle.namelist()))


In [ ]:
import subprocess, sys

# Preserve Kaggle's shared NumPy/SciPy stack; install only runtime libraries.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'lightgbm==4.7.0', 'rapidfuzz==3.14.3', 'Unidecode==1.4.0', 'catboost==1.2.10'
], check=True)
import numpy, scipy, lightgbm, rapidfuzz, unidecode, catboost
print('Runtime:', 'numpy', numpy.__version__, '| lightgbm', lightgbm.__version__,
      '| catboost', catboost.__version__)


## Attach and verify the correct dataset

Use Kaggle **Add Input** to attach the official challenge input, either as extracted `dataset/train` + `dataset/test` folders or as the `6ab10eb3b23ba_student_resource.zip` archive. This cell searches recursively and refuses to continue unless it finds exactly one complete train/test pair. It does not download from GitHub or require a GitHub token.


In [ ]:
from pathlib import Path, PurePosixPath
import zipfile

INPUT_ROOT = Path('/kaggle/input')
TRAIN_REQUIRED = {'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'}
TEST_REQUIRED = {'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv'}

def find_attached_dataset(search_roots=(INPUT_ROOT,)):
    found = []
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for source1 in search_root.rglob('train_source1.tsv'):
            train_dir = source1.parent
            test_dir = train_dir.parent / 'test'
            train_names = {p.name for p in train_dir.iterdir() if p.is_file()}
            test_names = {p.name for p in test_dir.iterdir() if p.is_file()} if test_dir.is_dir() else set()
            if TRAIN_REQUIRED.issubset(train_names) and TEST_REQUIRED.issubset(test_names):
                found.append((train_dir, test_dir))
    return found

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            rel = PurePosixPath(member.filename.replace('\\', '/'))
            if rel.is_absolute() or '..' in rel.parts:
                raise RuntimeError(f'Unsafe path in attached archive: {member.filename}')
            target = (destination / Path(*rel.parts)).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'Archive member escapes extraction folder: {member.filename}')
        zf.extractall(destination)

matches = find_attached_dataset()
print('Kaggle inputs:', [str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else 'missing')
if not matches and INPUT_ROOT.exists():
    archives = [p for p in INPUT_ROOT.rglob('*.zip') if 'student_resource' in p.name.lower()]
    for archive in archives:
        with zipfile.ZipFile(archive) as zf:
            names = {Path(n.replace('\\', '/')).name for n in zf.namelist()}
        if not (TRAIN_REQUIRED | TEST_REQUIRED).issubset(names):
            continue
        extracted = Path('/kaggle/working/challenge_data_from_input')
        safe_extract(archive, extracted)
        matches = find_attached_dataset((INPUT_ROOT, extracted))
        print('Extracted attached archive:', archive)
        if matches:
            break
if len(matches) != 1:
    raise RuntimeError(
        f'Expected exactly one complete dataset; found {len(matches)}. '
        'Attach the official archive or extracted dataset/train and dataset/test with Add Input.'
    )
TRAIN_DIR, TEST_DIR = matches[0]
print('TRAIN_DIR:', TRAIN_DIR)
print('TEST_DIR: ', TEST_DIR)
for p in sorted(TRAIN_DIR.iterdir()):
    if p.is_file():
        print(f'{p.name}: {p.stat().st_size / 1e9:.2f} GB')
for name in sorted(TRAIN_REQUIRED):
    with open(TRAIN_DIR / name, encoding='utf-8') as f:
        print(name, 'header:', f.readline().rstrip()[:180])


## Confirm accelerator and set the run configuration

The code uses the GPU for CatBoost fitting and, when the small probe succeeds, batched CatBoost inference; it uses the CPU for high-volume blocking and similarity calculations. LightGBM remains CPU-based. Kaggle T4 x2 does not make those Python candidate/feature stages GPU work. Training pair tables checkpoint every 50,000 Source-1 rows and test inference checkpoints every 1,024 rows. Re-running a cell after an interruption resumes from its last committed batch if `/kaggle/working` is still present. The inference implementation batches prediction calls to avoid millions of tiny CatBoost/LightGBM calls.


In [ ]:
import os, subprocess
from pathlib import Path
from catboost.utils import get_gpu_device_count

try:
    subprocess.run(['nvidia-smi', '-L'], check=False)
except FileNotFoundError:
    print('nvidia-smi is unavailable; checking CatBoost CUDA runtime directly.')
GPU_COUNT = get_gpu_device_count()
print('GPUs visible to CatBoost:', GPU_COUNT)
if GPU_COUNT < 1:
    raise RuntimeError('No CUDA GPU is visible. Set Kaggle Accelerator to GPU (T4 x2) and restart the session.')
GPU_DEVICES = '0:1' if GPU_COUNT >= 2 else '0'

# Fail fast if Kaggle exposes the devices but CatBoost cannot fit on them.
import numpy as np
_probe_rng = np.random.default_rng(2026)
_probe_x = _probe_rng.normal(size=(2048, 16)).astype('float32')
_probe_y = (_probe_x[:, 0] + _probe_x[:, 1] > 0).astype('uint8')
_gpu_probe = catboost.CatBoostClassifier(
    iterations=4, depth=4, task_type='GPU', devices=GPU_DEVICES,
    verbose=False, allow_writing_files=False, border_count=32
)
_gpu_probe.fit(_probe_x, _probe_y)
print('CatBoost GPU fit probe passed.')
try:
    _probe_probability = _gpu_probe.predict_proba(_probe_x, task_type='GPU', verbose=False)
    if _probe_probability.shape != (len(_probe_x), 2):
        raise RuntimeError(f'Unexpected GPU probability output shape: {_probe_probability.shape}')
    CATBOOST_PREDICTION_TASK = 'GPU'
    print('CatBoost GPU prediction probe passed; batched inference will use GPU.')
except Exception as exc:
    CATBOOST_PREDICTION_TASK = 'CPU'
    print('CatBoost GPU prediction probe failed; inference will use CPU:', repr(exc))
del _gpu_probe, _probe_x, _probe_y, _probe_rng

WORK_DIR = Path('/kaggle/working/v8_best_gpu')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_CANDIDATES_PER_SOURCE = 128
MAX_NAME_BLOCK_SIZE = 200
MAX_ADDRESS_BLOCK_SIZE = 1200
HARD_NEGATIVES_PER_SOURCE = 8
HOLDOUT_PER_MILLE = 10       # deterministic 1% of S1 entities; re-added for final fit
CATBOOST_ROUNDS = 1200       # early stopping uses held-out candidate-pair AUC
LIGHTGBM_ROUNDS = 1200       # CPU comparator; early stopping uses same holdout
EARLY_STOPPING_ROUNDS = 100
PAIR_CHECKPOINT_ROWS = 50_000
THREADS = min(4, max(1, os.cpu_count() or 2))

print('CatBoost devices:', GPU_DEVICES, '| CatBoost prediction:', CATBOOST_PREDICTION_TASK, '| CPU threads:', THREADS, '| pair checkpoint rows:', PAIR_CHECKPOINT_ROWS)
print('Per-source cap:', MAX_CANDIDATES_PER_SOURCE,
      '| address-only block cap:', MAX_ADDRESS_BLOCK_SIZE)


## Full-data training, model comparison, and threshold selection

This cell will run on the full training files. It first builds the shared capped candidate/feature tables once, trains CatBoost on the detected GPU(s) and LightGBM on CPU, compares both and three blends on the same held-out Source-1 IDs, and selects source-specific probability thresholds under target-record exclusivity. It then refits the selected model or blend after appending the holdout pairs, so the shipped model uses all labeled Source-1 entities.


In [ ]:
import json, subprocess, sys

TRAIN_SCRIPT = REPO_DIR / 'code/business_entity_resolution/src/train_v8_gpu.py'
REPORT_PATH = WORK_DIR / 'validation_report_v8.json'
META_PATH = WORK_DIR / 'entity_matcher_v8.json'
PAIR_CACHE_PATH = WORK_DIR / 'pair_tables_v8_manifest.json'
train_cmd = [
    sys.executable, str(TRAIN_SCRIPT),
    '--train-dir', str(TRAIN_DIR), '--work-dir', str(WORK_DIR),
    '--max-candidates-per-source', str(MAX_CANDIDATES_PER_SOURCE),
    '--max-block-size', str(MAX_NAME_BLOCK_SIZE),
    '--max-address-block-size', str(MAX_ADDRESS_BLOCK_SIZE),
    '--hard-negatives-per-source', str(HARD_NEGATIVES_PER_SOURCE),
    '--holdout-per-mille', str(HOLDOUT_PER_MILLE),
    '--catboost-rounds', str(CATBOOST_ROUNDS),
    '--lightgbm-rounds', str(LIGHTGBM_ROUNDS),
    '--early-stopping-rounds', str(EARLY_STOPPING_ROUNDS),
    '--checkpoint-rows', str(PAIR_CHECKPOINT_ROWS),
    '--depth', '7', '--learning-rate', '0.05', '--l2-leaf-reg', '5.0',
    '--border-count', '64', '--devices', GPU_DEVICES,
    '--threads', str(THREADS), '--verbose', '100'
]
expected_train_config = {
    'catboost_rounds_requested': CATBOOST_ROUNDS,
    'lightgbm_rounds_requested': LIGHTGBM_ROUNDS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 5.0,
    'border_count': 64, 'metric_period': 25, 'devices': GPU_DEVICES, 'seed': 2026,
    'threads': THREADS, 'holdout_per_mille': HOLDOUT_PER_MILLE,
    'checkpoint_rows': PAIR_CHECKPOINT_ROWS,
}
reuse_complete_run = False
if META_PATH.is_file() and REPORT_PATH.is_file() and PAIR_CACHE_PATH.is_file():
    previous = json.loads(META_PATH.read_text(encoding='utf-8'))
    pair_cache = json.loads(PAIR_CACHE_PATH.read_text(encoding='utf-8'))
    expected_caps = (
        previous.get('max_candidates_per_source') == MAX_CANDIDATES_PER_SOURCE and
        previous.get('max_block_size') == MAX_NAME_BLOCK_SIZE and
        previous.get('max_address_block_size') == MAX_ADDRESS_BLOCK_SIZE and
        previous.get('hard_negatives_per_source') == HARD_NEGATIVES_PER_SOURCE
    )
    expected_models_exist = all((WORK_DIR / name).is_file()
                                for name in previous.get('model_files', {}).values())
    reuse_complete_run = (
        expected_caps and expected_models_exist and
        previous.get('training_config') == expected_train_config and
        previous.get('pair_table_config') == pair_cache.get('config')
    )
if reuse_complete_run:
    print('A complete matching V8 model/report already exists; reusing it to avoid an accidental full retrain.')
else:
    print('Running:', ' '.join(train_cmd), flush=True)
    subprocess.run(train_cmd, cwd=REPO_DIR, check=True)
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2))


## Score ceiling review and full test inference

Check `candidate_oracle_macro_f05` first. This is the best possible macro F0.5 if every retrieved true pair were predicted and every retrieved false pair rejected. If the oracle is below 0.99, the 0.99 target is mathematically out of reach for this candidate configuration; the selected classifier still gives the best measured V8 validation option. The full inference cell stays enabled to produce the strongest available submission unless you set `RUN_FULL_TEST_INFERENCE = False`.


In [ ]:
TARGET_F05 = 0.99
oracle_f05 = float(report['candidate_oracle_macro_f05'])
model_f05 = float(report['macro_f05'])
print(f"Candidate-only ceiling: {oracle_f05:.6f}; selected validation model: {model_f05:.6f}; target: {TARGET_F05:.2f}")
if oracle_f05 < TARGET_F05:
    print('CEILING WARNING: candidate retrieval must improve before this pipeline can reach the 0.99 target.')
elif model_f05 < TARGET_F05:
    print('MODEL GAP: candidate recall permits the target, but the current matcher/threshold does not reach it.')
else:
    print('The validation target was reached; this is still a held-out estimate, not a leaderboard guarantee.')

RUN_FULL_TEST_INFERENCE = True
OUT_DIR = Path('/kaggle/working/output_v8_best')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_FULL_TEST_INFERENCE:
    PREDICT_SCRIPT = REPO_DIR / 'code/business_entity_resolution/src/predict_v8_gpu.py'
    infer_cmd = [
        sys.executable, str(PREDICT_SCRIPT), '--test-dir', str(TEST_DIR),
        '--model-dir', str(WORK_DIR), '--output-dir', str(OUT_DIR),
        '--query-batch-size', '1024', '--prediction-threads', '2',
        '--catboost-prediction-task', CATBOOST_PREDICTION_TASK
    ]
    print('Running:', ' '.join(infer_cmd), flush=True)
    subprocess.run(infer_cmd, cwd=REPO_DIR, check=True)
else:
    print('Full test inference was disabled by configuration.')


## Validate and package the submission outputs

The challenge validator does a streaming format/subset check without loading every target ID into a large in-memory set. The optional `--check-ids` mode is intentionally omitted here to keep RAM available on Kaggle; the prediction writer itself only emits IDs from its indexed test source tables.


In [ ]:
import zipfile

if RUN_FULL_TEST_INFERENCE:
    MATCHING_PATH = OUT_DIR / 'matching_results.tsv'
    CANDIDATE_PATH = OUT_DIR / 'candidate_pairs.tsv'
    VALIDATOR = REPO_DIR / 'utils/validate_submission.py'
    subprocess.run([
        sys.executable, str(VALIDATOR), '--matching', str(MATCHING_PATH),
        '--candidate', str(CANDIDATE_PATH), '--test-dir', str(TEST_DIR)
    ], cwd=REPO_DIR, check=True)

    SUBMISSION_ZIP = OUT_DIR / 'submission_v8_best.zip'
    with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        zf.write(MATCHING_PATH, arcname='matching_results.tsv')
        zf.write(CANDIDATE_PATH, arcname='candidate_pairs.tsv')
    for path in (MATCHING_PATH, CANDIDATE_PATH, SUBMISSION_ZIP):
        print(path.name, f'{path.stat().st_size / 1e9:.3f} GB')
else:
    print('Inference was disabled; there are no submission files to validate/package.')
